# 31 — Post-shift opto: the dispensability test

The expert/uniform result (NB30) is a *predicted* null: PPC should be dispensable
once the animal's model is adequate. This notebook runs the decisive test — does
silencing PPC matter **right after a distribution shift**, when the model is wrong
and must be updated?

The logic has two parts:

1. **Confirmation (§2)** — does silencing move behaviour post-shift *at all*?
   This is the gate the whole story rests on: a null in the expert phase only
   means "dispensable" if we can show the laser *can* move behaviour where it
   should. HET vs WT on the post-shift Δ; this is the better-powered test (one Δ,
   6-vs-4, floors ~p≈0.0095).
2. **Dispensability interaction (§3)** — does the opto effect *grow* from expert
   to post-shift? Per animal, Δ_post-shift − Δ_expert. This is the title's claim,
   but it is a difference-of-differences and only uses animals with both phases,
   so expect **underpowered / directional**, not definitive (within-HET Wilcoxon
   at n=6 floors at p=0.031).

**Pooling Hard-A and Hard-B.** They are mirror distributions. The history stats
(recency, win/lose) are invariant to the mirror, so they pool — that is the
'hard' (pooled A+B) view, and the headline leads with recency. The lateralised
curve params (PSE, lapse) shift in *opposite* directions across the mirror, so
pooling cancels them; those are read **per phase only** (§3b), and the update
matrices (§5) keep A and B separate for the same reason.

Unit of inference is the animal throughout (pool within animal → one value per
animal → test across animals). Genotype is injected (the loader doesn't carry it).


In [ ]:
from shared_setup import *
from analysis.opto import compute_opto_stats, compute_opto_delta
from analysis.phase import compute_group_phase
from plotting.opto import plot_delta_swarm, plot_delta_paired
from behav_utils.plotting import plot_um
from scipy.stats import mannwhitneyu, wilcoxon
from IPython.display import display

experiment, info = load_data()

# genotype is not populated by the loader — inject the opto cohort (same map as NB30)
GENOTYPE = {'SS14': 'wt', 'SS17': 'wt', 'SS18': 'wt', 'SS20': 'wt',
            'SS15': 'het', 'SS16': 'het', 'SS19': 'het',
            'SS21': 'het', 'SS22': 'het', 'SS23': 'het'}
for aid, a in experiment.animals.items():
    if aid in GENOTYPE:
        a.metadata['genotype'] = GENOTYPE[aid]
WT_IDS   = [a for a in GENOTYPE if GENOTYPE[a] == 'wt'  and a in experiment.animals]
HET_IDS  = [a for a in GENOTYPE if GENOTYPE[a] == 'het' and a in experiment.animals]
OPTO_IDS = WT_IDS + HET_IDS

POOL_STATS   = ['recency', 'win_stay', 'lose_shift']   # symmetric -> valid to pool A+B
LAT_STATS    = ['pse', 'slope', 'lapse']               # lateralised/curve -> per phase only
LEAD         = 'recency'                               # robust, symmetric -> headline
N_BOOT_CURVE = 200                                     # curve-param CI (the reliability gate)
PHASES       = {'uniform': 'Uniform', 'hard_a': 'Hard-A', 'hard_b': 'Hard-B'}

print(f"HET ({len(HET_IDS)}): {HET_IDS}")
print(f"WT  ({len(WT_IDS)}): {WT_IDS}")
if not HET_IDS or not WT_IDS:
    print("One genotype group is empty — check the GENOTYPE map against loaded animals.")

In [ ]:
def mannwhit(delta_df, stat, phase, a='het', b='wt'):
    """Mann-Whitney on Δ between genotypes within one phase. Returns (p, n_a, n_b)."""
    d = delta_df[(delta_df['stat'] == stat) & (delta_df['phase'] == phase)]
    va = d[d['genotype'] == a]['delta'].dropna()
    vb = d[d['genotype'] == b]['delta'].dropna()
    if len(va) < 1 or len(vb) < 1:
        return np.nan, len(va), len(vb)
    try:
        p = mannwhitneyu(va, vb, alternative='two-sided').pvalue
    except ValueError:
        p = np.nan
    return p, len(va), len(vb)


def interaction(delta_df, stat, pa='uniform', pb='hard'):
    """Δ_pb vs Δ_pa: within-HET paired Wilcoxon + HET-vs-WT on (Δ_pb − Δ_pa).

    Paired set only (animals with both phases). Returns a dict of p-values and ns.
    """
    w = (delta_df[delta_df['stat'] == stat]
         .pivot_table(index=['animal', 'genotype'], columns='phase', values='delta'))
    if pa not in w.columns or pb not in w.columns:
        return dict(stat=stat, within_het_p=np.nan, het_vs_wt_p=np.nan,
                    n_paired_het=0, n_paired_wt=0)
    w = w.dropna(subset=[pa, pb]).reset_index()
    w['D'] = w[pb] - w[pa]
    h, wt = w[w['genotype'] == 'het'], w[w['genotype'] == 'wt']
    if len(h) >= 1 and (h[pa].to_numpy() != h[pb].to_numpy()).any():
        try:
            wp = wilcoxon(h[pb], h[pa]).pvalue
        except ValueError:
            wp = np.nan
    else:
        wp = np.nan
    if len(h) >= 1 and len(wt) >= 1:
        try:
            mp = mannwhitneyu(h['D'], wt['D'], alternative='two-sided').pvalue
        except ValueError:
            mp = np.nan
    else:
        mp = np.nan
    return dict(stat=stat, within_het_p=wp, het_vs_wt_p=mp,
                n_paired_het=len(h), n_paired_wt=len(wt))

## §1 — Aggregate Δ-stats across phases

Symmetric history stats for uniform / Hard-A / Hard-B / pooled `hard`, and the
lateralised curve params for the three phases separately. One Δ per animal per
stat per phase; the pooled `hard` is a session-level pool of A+B (so it is
properly trial-weighted), not an average of the two deltas.

In [ ]:
SYM_PHASES = {'uniform': 'uniform', 'hard_a': 'hard_a', 'hard_b': 'hard_b',
              'hard': ['hard_a', 'hard_b']}          # 'hard' = session-level A+B pool

sym_stats, sym = {}, []
for name, ph in SYM_PHASES.items():
    s = compute_opto_stats(experiment, phase=ph, stats=POOL_STATS,
                           animals=OPTO_IDS, n_bootstrap=0)
    sym_stats[name] = s
    sym.append(compute_opto_delta(s).assign(phase=name))
delta_sym = pd.concat(sym, ignore_index=True)

ORDER = [LEAD] + [s for s in POOL_STATS if s != LEAD]
print('Δ', LEAD, 'per animal across phases:')
display(delta_sym[delta_sym['stat'] == LEAD]
        .pivot_table(index=['animal', 'genotype'], columns='phase', values='delta')
        .round(3))

### §1b — Lateralised curve params (slow cell)

This fits the psychometric per condition with the bootstrap CI (`N_BOOT_CURVE`)
across all three phases — it is the slow step. §2 and §3 (the headline history
stats) don't depend on it, so you can run those first; only §3b needs `delta_lat`.
Lower `N_BOOT_CURVE` to iterate.

In [ ]:
LAT_PHASES = {'uniform': 'uniform', 'hard_a': 'hard_a', 'hard_b': 'hard_b'}
lat = []
for name, ph in LAT_PHASES.items():
    s = compute_opto_stats(experiment, phase=ph, stats=LAT_STATS,
                           animals=OPTO_IDS, n_bootstrap=N_BOOT_CURVE)
    lat.append(compute_opto_delta(s).assign(phase=name))
delta_lat = pd.concat(lat, ignore_index=True)
print('lateralised Δ ready for:', sorted(delta_lat['stat'].unique()))

## §2 — Confirmation: does silencing move behaviour post-shift?

Pooled Hard-A+B, Δ (opto − nonopto), HET vs WT. If this is a flat null in HET as
well, the expert null is uninterpretable (the laser may simply do nothing) — so
this is the gate, and recency (robust) leads. The per-phase A/B swarms below check
the pooling assumption: for symmetric stats they should look alike.

In [ ]:
conf = pd.DataFrame([dict(stat=st, **dict(zip(['p_HET_vs_WT', 'n_het', 'n_wt'],
                                              mannwhit(delta_sym, st, 'hard'))))
                     for st in ORDER])
display(conf.round(4))

fig, axes = plt.subplots(1, len(ORDER), figsize=(3.4 * len(ORDER), 3.6))
for ax, st in zip(np.atleast_1d(axes), ORDER):
    p = conf.loc[conf['stat'] == st, 'p_HET_vs_WT'].iloc[0]
    plot_delta_swarm(delta_sym[delta_sym['phase'] == 'hard'], st, ax=ax, p_value=p)
fig.suptitle('Post-shift (pooled Hard-A+B): Δ (opto − nonopto), HET vs WT', y=1.03)
fig.tight_layout(); plt.show()

In [ ]:
# symmetry check — same stats, Hard-A and Hard-B separately
for phase in ['hard_a', 'hard_b']:
    fig, axes = plt.subplots(1, len(ORDER), figsize=(3.4 * len(ORDER), 3.4))
    for ax, st in zip(np.atleast_1d(axes), ORDER):
        p, _, _ = mannwhit(delta_sym, st, phase)
        plot_delta_swarm(delta_sym[delta_sym['phase'] == phase], st, ax=ax, p_value=p)
    fig.suptitle(f'{PHASES[phase]}: Δ (opto − nonopto), HET vs WT', y=1.03)
    fig.tight_layout(); plt.show()

## §3 — The dispensability interaction: Δ expert vs Δ post-shift

Per animal, does the opto effect grow from expert to post-shift? `within_het_p`
is the paired Wilcoxon across HET animals (Δ_hard vs Δ_uniform); `het_vs_wt_p`
tests whether that growth is specific to inhibition (HET) vs the light artifact
(WT), via the difference-of-differences. Both are underpowered and use only the
paired set (`n_paired_*`) — read this as direction-of-effect, not proof. The
paired plot shows each animal's expert→post-shift Δ; a consistent steepening in
HET (and not WT) is the signature we're after.

In [ ]:
inter = pd.DataFrame([interaction(delta_sym, st, 'uniform', 'hard') for st in ORDER])
display(inter.round(4))

fig, axes = plt.subplots(1, len(ORDER), figsize=(3.9 * len(ORDER), 3.8))
for ax, st in zip(np.atleast_1d(axes), ORDER):
    p = inter.loc[inter['stat'] == st, 'within_het_p'].iloc[0]
    plot_delta_paired(delta_sym, st, ax=ax, phase_a='uniform', phase_b='hard', p_value=p)
fig.suptitle('Dispensability: per-animal Δ expert → Δ post-shift (pooled hard)', y=1.03)
fig.tight_layout(); plt.show()

## §3b — Lateralised curve params (per phase, not pooled)

PSE, slope and lapse from the curve fit. PSE and lapse are lateralised, so the
pooled A+B view would cancel them — they are shown for uniform / Hard-A / Hard-B
separately. These rest on the per-condition curve fit, which is the thin, less
reliable readout post-shift (see §6); treat as secondary to the history stats.

In [ ]:
for phase in ['uniform', 'hard_a', 'hard_b']:
    fig, axes = plt.subplots(1, len(LAT_STATS), figsize=(3.4 * len(LAT_STATS), 3.4))
    for ax, st in zip(np.atleast_1d(axes), LAT_STATS):
        p, _, _ = mannwhit(delta_lat, st, phase)
        plot_delta_swarm(delta_lat[delta_lat['phase'] == phase], st, ax=ax, p_value=p)
    fig.suptitle(f'{PHASES[phase]}: Δ curve params, HET vs WT  (per phase — not pooled)', y=1.03)
    fig.tight_layout(); plt.show()

## §3c — Curve-param dispensability (paired, per hard phase)

The same expert→post-shift paired view as §3, for the curve params. Hard-A and
Hard-B are mirror distributions, so an opto-induced PSE (or lapse) shift can go in
*opposite* directions across them — the pooled `hard` used in §3 would cancel it.
So here the interaction is run per hard phase: uniform → Hard-A and uniform →
Hard-B, separately. Slope is symmetric, so its two panels should agree (a free
check); PSE and lapse may legitimately differ between A and B. All three rest on
the per-condition curve fit — the thin readout post-shift (see §6) — so read as
directional at most.

In [ ]:
HARD = ['hard_a', 'hard_b']
lat_inter = pd.DataFrame([{'hard_phase': ph, **interaction(delta_lat, st, 'uniform', ph)}
                          for st in LAT_STATS for ph in HARD])
display(lat_inter.round(4))

fig, axes = plt.subplots(len(LAT_STATS), len(HARD),
                         figsize=(4.1 * len(HARD), 3.7 * len(LAT_STATS)), squeeze=False)
for i, st in enumerate(LAT_STATS):
    for j, ph in enumerate(HARD):
        p = lat_inter[(lat_inter['stat'] == st) &
                      (lat_inter['hard_phase'] == ph)]['within_het_p'].iloc[0]
        plot_delta_paired(delta_lat, st, ax=axes[i][j],
                          phase_a='uniform', phase_b=ph, p_value=p)
        axes[i][j].set_title(f"{st}: uniform → {PHASES[ph]}", fontsize=10)
fig.suptitle('Curve-param dispensability: Δ expert → Δ post-shift, per hard phase', y=1.005)
fig.tight_layout(); plt.show()

## §5 — Post-shift update matrices

Group-mean update matrices, nonopto / opto / difference, per genotype, Hard-A and
Hard-B separate (mirror distributions — pooling would blur the structure).
Mega-pooled across the group given how thin post-shift opto trials are. The
difference matrix carries the noise of both, so read it for gross structure only;
the SC→BE-reversion prediction lives here but needs more trials to test.

In [ ]:
for phase in ['hard_a', 'hard_b']:
    for label, ids in [('HET', HET_IDS), ('WT', WT_IDS)]:
        if not ids:
            continue
        _, um = compute_group_phase(experiment, phase, ids, method='pooled', n_bootstrap=0)
        u_off, u_on = um.get('opto_off'), um.get('opto_on')
        fig, axes = plt.subplots(1, 3, figsize=(10.5, 3.4))
        for ax, u, ttl in [(axes[0], u_off, 'nonopto'), (axes[1], u_on, 'opto')]:
            if u is not None:
                plot_um(u, ax=ax); ax.set_title(f'{label} {PHASES[phase]} — {ttl}', fontsize=9)
            else:
                ax.set_visible(False)
        if u_on is not None and u_off is not None:
            diff = np.asarray(u_on['um'], float) - np.asarray(u_off['um'], float)
            vmax = float(np.nanmax(np.abs(diff))) or 1.0
            plot_um(diff, ax=axes[2], vmin=-vmax, vmax=vmax)
            axes[2].set_title(f'{label} — Δ (opto − nonopto)', fontsize=9)
        else:
            axes[2].set_visible(False)
        fig.tight_layout(); plt.show()

## §6 — Data check: trial counts per phase × condition

How thin is post-shift? Trials per animal per condition (opto / nonopto / post),
for each phase. This is the honest backdrop to the power caveats above — the tests
in §2–§3 are only as strong as these counts allow.

In [ ]:
qc = pd.concat([sym_stats[name][sym_stats[name]['stat'] == LEAD].assign(phase=name)
                for name in ['uniform', 'hard_a', 'hard_b']], ignore_index=True)
display(qc.pivot_table(index=['genotype', 'animal'], columns=['phase', 'condition'],
                       values='n_trials', fill_value=0).astype(int))